# 第3讲：QLoRA 微调 LLaMA-3——面向数字人文术语的领域适配

第1讲中，我们加载了 LLaMA-3 并用它对短视频转写文本做自动摘要；第2讲中，我们用 CLIP 的跨模态编码对视频帧与古籍原文片段进行了相似度对齐。
然而，通用预训练模型对“刻本”、“缮写”、“金石学”这类数字人文专业术语往往语焉不详，甚至给出过度泛化的解释。

**本讲的目标**：通过 **QLoRA**（量化低秩适配）对 LLaMA-3 做轻量级领域微调，使它在读入古籍转写片段后，能够给出准确、可复现的专业术语解释——这是构建数字人文专属知识助手的关键一步。

# 本讲结构

## 1  环境准备与 GPU 自检
## 2  4 位量化加载 LLaMA-3（BitsAndBytes NF4）
## 3  检查点 1：确认基础模型正常运行
## 4  LoRA 参数高效微调配置
## 5  构建占位数据集（10 条样本）与分词
## 6  检查点 2：在小数据上确认损失下降
## 7  解读损失曲线
## 8  上传真实数据 → 全量语料训练
## 9  保存适配器 → 下载到本地（供第 4 讲复用）
## 快速回顾

# 透明度说明

> **模型来源**：本讲使用 `NousResearch/Meta-Llama-3-8B-Instruct`，这是 Meta 官方 Llama 3 权重由社区托管在 Hugging Face 上的**镜像版本**，无需 Hugging Face 账号或访问令牌即可直接下载。原始模型授权见 Meta Llama 3 Community License Agreement。

# 第1章  环境准备与 GPU 自检

- 安装四个关键库：`transformers`（模型）、`peft`（LoRA）、`bitsandbytes`（4位量化）、`datasets`（数据集）
- 运行 GPU 自检——如果没有 GPU，脚本会立即报错并给出明确提示，而不是悄悄在 CPU 上运行

> **本地用户（mllm-dh 环境）**：如果已在 `mllm-dh` conda 环境中运行，可跳过 `!pip install` 行，直接执行 GPU 自检部分。

In [ ]:
# ===== 第1章：环境准备 =====
# Colab 用户：运行此单元格自动安装依赖。
# 本地 mllm-dh 环境用户：可注释掉 !pip 行，直接运行 GPU 自检。

!pip install -q transformers accelerate peft bitsandbytes datasets

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        '未检测到 GPU。请在 Colab 中选择：运行时 → 更改运行时类型 → T4 GPU。'
        ' 本讲的 4 位量化模型无法在 CPU 上有效运行。'
    )

device_name = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU 已就绪：{device_name}')
print(f'显存总量：{total_mem:.1f} GB')

# 第2章  4 位量化加载 LLaMA-3

## 为什么要“量化”？

LLaMA-3 8B 的全精度权重（fp32）需要约 **32 GB** 显存——远超 Colab 免费 GPU 的上限。
**4 位 NF4 量化**把每个权重从 32 位压缩至 4 位，将显存需求降至约 **5–6 GB**，同时用“双重量化”进一步压缩量化常数自身的存储开销。

> 直觉：量化是一种有损压缩，但在大多数推理和微调任务中，精度损失对最终结果的影响甚微。

In [ ]:
# ===== 第2章：加载分词器与量化配置 =====
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'NousResearch/Meta-Llama-3-8B-Instruct'

# 根据 GPU 能力选择计算精度（T4 及以下用 fp16；Ampere 及以上用 bf16）
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'计算精度：{compute_dtype}')

# 4 位量化配置：NF4 格式 + 双重量化
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

# 加载分词器——Llama-3 默认没有 pad_token，必须显式设置
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('分词器加载完成')

In [ ]:
# 加载 4 位量化模型
# 首次运行需下载约 5 GB 权重，Colab 大约需要 3–5 分钟，请耐心等待
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.config.use_cache = False  # 微调阶段禁用 KV cache 以节省显存

print('模型加载完成')
print(f'参数量：{sum(p.numel() for p in model.parameters()) / 1e9:.2f} B')

# 第3章  检查点 1：确认基础模型正常运行

在配置任何 LoRA 之前，先做一次推理验证。

这是一个**有意设置的检查点**：如果基础模型加载有问题（权重损坏、量化参数不兼容等），这里会立即报错，避免在后续的 LoRA 配置和训练上浪费时间。

In [ ]:
# ===== 检查点 1：确认基础模型正常运行，再触碰 LoRA =====

test_prompt = '请用一句话解释什么是数字人文学。'
messages = [{'role': 'user', 'content': test_prompt}]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(formatted, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== 检查点 1 输出 ===')
print(response)
print('=' * 22)
print('基础模型正常，可以继续配置 LoRA。')

# 第4章  LoRA：参数高效微调的直觉

## 为什么不全量微调？

全量微调需要为所有 80 亿参数存储梯度和优化器状态，显存需求超过 60 GB。

**LoRA 的思路**：冻结原模型权重，仅在注意力层的关键线性矩阵旁插入两个小的低秩矩阵（A 和 B，其中 rank = r）。训练时只更新这两个小矩阵，推理时将它们的乘积“加回”原权重。

最终可训练参数通常不到原模型的 **1%**，但足以使模型适应新的领域词汇和表达风格。

In [ ]:
# ===== 第4章：LoRA 配置 =====
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# 为 4 位量化训练准备模型（启用梯度检查点、调整层归一化精度）
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                       # 低秩矩阵的秩（越大=容量越高，但显存更多）
    lora_alpha=32,              # 缩放因子，通常设为 2×r
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,          # 轻度 dropout 防止过拟合
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'可训练参数：{trainable:,}')
print(f'总参数：{total:,}')
print(f'可训练比例：{100 * trainable / total:.4f}%')

# 第5章  构建占位数据集

## 数据格式：指令跟随（Instruction Following）

每条训练样本由三部分组成，遵循 Llama-3 的对话模板：

| 角色 | 内容 |
|------|------|
| system | 告诉模型它是数字人文领域的专家 |
| user | 视频转写片段（包含待解释的术语） |
| assistant | 准确的专业术语解释（训练目标） |

以下 10 条样本是**占位示例**，涵盖版本学、档案学、金石学等核心术语，需在正式训练前替换为你的真实视频转写数据。

In [ ]:
# ===== 第5章：构建占位数据集（10 条）=====
# ⚠️  以下为占位示例，请在正式训练前替换为真实的视频转写数据。

SYSTEM_PROMPT = (
    '你是一位专精于数字人文领域的学者，'
    '能够准确解释古籍文献、文化遗产与档案实践中的专业术语。'
)

RAW_DATA = [
    {
        'transcript': '视频中提到这部书是刻本，与手写本不同。',
        'explanation': '刻本（kè běn）指雕版印刷的书籍：工匠将文字反刻于木版，涂墨后压印于纸张。宋元刻本是版本学研究的核心文物。',
    },
    {
        'transcript': '文献里反复出现缮写一词，请问这是什么意思？',
        'explanation': '缮写指专职抄手对文本进行誊清、校正的抄录工序，是印刷术普及前文本传播的主要手段，也是研究字形演变的重要证据。',
    },
    {
        'transcript': '研究者说他的研究方向是金石学。',
        'explanation': '金石学是中国传统学问中研究铜器铭文（金）与石刻碑文（石）的学科，是现代铭刻学和考古学的前身，为历史地理与文字演变研究提供第一手资料。',
    },
    {
        'transcript': '档案整理时需要遵循原始顺序原则。',
        'explanation': '原始顺序原则（Principle of Original Order）是档案学核心原则之一：档案应按照其创建机构或个人最初形成时的顺序保存，以保留内在的背景关系与证据价值。',
    },
    {
        'transcript': '数字化之后，团队对每张图片做了标注。',
        'explanation': '数字化标注（digital annotation）指在数字化图像或文本上附加结构化元数据（如人名、地名、朝代、主题词），以支持计算机检索与人文分析。',
    },
    {
        'transcript': '不同版本之间存在异文，需要做校勘。',
        'explanation': '校勘（jiào kān）是文献学的核心方法：通过比对同一文本的不同传本，发现并考证文字差异（异文），以期恢复最接近原始面貌的文字。',
    },
    {
        'transcript': '这是一部传世本，与出土文献有所不同。',
        'explanation': '传世本指通过历代抄写、刻印而连续流传至今的文本版本；出土文献则指通过考古发掘获得的实物文本（如简牍、帛书）。二者互校是文献学研究的重要方法。',
    },
    {
        'transcript': '博物馆里展出了多件拓片。',
        'explanation': '拓片（碑帖）是将宣纸覆于石刻或铜器表面，用墨捶拓而成的平面复制品，是传递金石文字最重要的媒介，也是数字化采集的优先对象。',
    },
    {
        'transcript': '他的博士论文研究的是目录学。',
        'explanation': '目录学（bibliography）研究文献的著录、分类与目录编制方法，在中国学术传统中有独立的学科地位，现代数字图书馆的元数据标准（如 MARC、Dublin Core）可视为其数字延伸。',
    },
    {
        'transcript': '视频介绍了一处文化遗产的数字孪生项目。',
        'explanation': '数字孪生（digital twin）在文化遗产语境下指通过三维扫描、摄影测量等技术建立遗址或文物的高精度数字模型，用于远程研究、虚拟展览与损毁预防。',
    },
]

print(f'数据集大小：{len(RAW_DATA)} 条')
print('示例条目（第 1 条）：')
print('  转写片段：', RAW_DATA[0]['transcript'])
print('  解释前50字：', RAW_DATA[0]['explanation'][:50], '...')

In [ ]:
# ===== 第5章（续）：分词函数 =====
from datasets import Dataset
from transformers import default_data_collator

MAX_LENGTH = 512

def format_and_tokenize(example):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': example['transcript']},
        {'role': 'assistant', 'content': example['explanation']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    tokenized = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
    )
    # 将填充位置的标签设为 -100，避免将 padding token 的预测纳入损失计算
    tokenized['labels'] = [
        -100 if mask == 0 else iid
        for iid, mask in zip(tokenized['input_ids'], tokenized['attention_mask'])
    ]
    return tokenized

hf_dataset = Dataset.from_list(RAW_DATA)
tokenized_dataset = hf_dataset.map(
    format_and_tokenize,
    remove_columns=['transcript', 'explanation'],
)

print(f'分词完成，样本数：{len(tokenized_dataset)}')
print(f'Token 序列长度：{len(tokenized_dataset[0]["input_ids"])}')

# 第6章  检查点 2：在小数据上确认损失下降

## 为什么先用 10 条样本训练？

10 条样本的训练目的**不是**让模型学会术语，而是验证完整的训练流程（数据 → 前向传播 → 反向传播 → 参数更新）没有错误。

**正常结果**：3 个 epoch 内，损失从约 2–3 稳定降至 1 以下。如果损失不动或出现异常，见下一章排查。

| 超参数 | 值 | 说明 |
|--------|-----|------|
| batch size | 1 | 低显存环境必须 |
| gradient accumulation | 4 | 等效 batch size = 4 |
| epochs | 3 | 仅用于流程验证 |
| learning rate | 2e-4 | LoRA 常用区间 |

In [ ]:
# ===== 检查点 2：在小数据上确认损失下降 =====
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./qlora_checkpoint',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=1,
    save_strategy='no',
    report_to='none',               # 不启用 wandb 等外部日志集成
    learning_rate=2e-4,
    warmup_steps=1,                     # 小数据集仅需 1 步预热
    lr_scheduler_type='cosine',
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=default_data_collator,
)

print('开始训练（检查点 2：小数据验证）...')
trainer.train()
print('训练完成。')

In [ ]:
# 打印损失历史（每步）
loss_history = [
    (entry['step'], entry['loss'])
    for entry in trainer.state.log_history
    if 'loss' in entry
]

print(f'{"步骤":>6}  {"损失":>10}')
print('-' * 20)
for step, loss in loss_history:
    print(f'{step:>6}  {loss:>10.4f}')

# 第7章  如何解读损失曲线

## 健康的下降曲线（10 条样本，3 个 epoch）

- 初始损失约 **2–3**（模型几乎随机预测）
- 损失平稳下降，最终降至 **0.5 以下**
- 曲线可能有波动，但整体方向向下

## 警告信号及最可能的原因

| 症状 | 最可能的原因 |
|------|-------------|
| 损失完全不动（平线） | `target_modules` 写错导致 LoRA 未真正注入；或学习率太低（< 1e-5） |
| 损失出现 NaN | 学习率太高（> 5e-3）；bf16/fp16 数值溢出 |
| 损失爆炸（快速增大） | 梯度裁剪未启用；批量过小导致梯度噪声过大 |
| 损失极低（< 0.01）但推理乱码 | 过拟合 10 条数据；标签/填充 masking 问题导致模型“背答案” |

> 提醒：10 条数据的训练目的是**验证流程**，而非追求最低损失。损失降了，流程就通了。

# 第8章  扩展至全量语料

## 从流程验证到真实训练

检查点 2 通过后，只需做两处替换即可扩展至真实数据集：
1. **数据来源**：从 JSON/CSV 文件加载，而不是硬编码列表
2. **Epoch 数量**：语料越大，所需 epoch 越少（避免过拟合）

**Colab 免费 GPU 时间预算参考：**

| 样本数 | 建议 epoch | 预估时长（T4）|
|--------|-----------|---------------|
| ≤ 500 条 | 3–5 | < 1 小时 |
| 500–2000 条 | 1–2 | 1–3 小时 |
| > 2000 条 | 1 + 保存检查点 | > 3 小时（可能需续接会话）|

# 第8章（续）  上传真实训练数据

在运行全量语料训练之前，请先将课程仓库中的 `shared/corpus/real_examples.json` 上传至 Colab。

该文件包含 **10 条真实的数字人文视频转写→术语解释对**，涵盖：甲骨文、古籍修复、非遗传承、敦煌壁画、活字印刷、青铜铭文、昆曲、简牍、拓片、二十四节气。

> 上传后，`DATA_PATH` 变量将自动更新，下方训练单元格可直接取消注释运行。

In [ ]:
import json
from google.colab import files

# 从本地电脑上传 real_examples.json
# 文件位于课程仓库的 shared/corpus/real_examples.json
print('请选择并上传 real_examples.json（位于课程仓库的 shared/corpus/ 目录下）')
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('未上传任何文件，请重新运行此单元格并选择 real_examples.json。')

DATA_PATH = list(uploaded.keys())[0]
print(f'已上传：{DATA_PATH}')

# 预览文件内容，确认上传正确
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    preview = json.load(f)

print(f'数据条数：{len(preview)}')
print('第 1 条转写片段：', preview[0]['transcript'])

In [ ]:
# ===== 第8章：全量语料训练模板 =====
# ⚠️  运行前请确认以下三步已完成：
#   步骤 1：检查点 2 已通过（损失正常下降）
#   步骤 2：已运行上方上传单元格，DATA_PATH 已指向 real_examples.json
#   步骤 3：取消注释下方 open(DATA_PATH) 和 full_trainer.train() 两处代码
# （json、Dataset、default_data_collator、TrainingArguments、Trainer 已在前面单元格导入）

# DATA_PATH 已由上方上传单元格设置，此处不再重新赋值
NUM_EPOCHS = 2       # 语料越大，epoch 越少
SAVE_PATH = './qlora_adapter_full'

# 步骤 3a：取消注释，加载真实数据
# with open(DATA_PATH, 'r', encoding='utf-8') as f:
#     full_raw_data = json.load(f)

# 占位：上方步骤 3a 注释时仍使用小数据集演示流程
full_raw_data = RAW_DATA

full_dataset = Dataset.from_list(full_raw_data)
full_tokenized = full_dataset.map(
    format_and_tokenize,
    remove_columns=['transcript', 'explanation'],
)
print(f'全量数据集大小：{len(full_tokenized)} 条')

full_training_args = TrainingArguments(
    output_dir=SAVE_PATH,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='epoch',              # 正式训练时保存每个 epoch 的检查点
    report_to='none',
    learning_rate=2e-4,
    warmup_steps=10,                    # 约占 10 样本×3 epoch 总步数的 3%
    lr_scheduler_type='cosine',
    remove_unused_columns=False,
)

full_trainer = Trainer(
    model=model,
    args=full_training_args,
    train_dataset=full_tokenized,
    data_collator=default_data_collator,
)

# 步骤 3b：取消注释，运行正式训练
# full_trainer.train()
# print('全量训练完成。')
print('（全量训练已注释——完成步骤 3a/3b 后即可运行）')

# 第9章  保存 LoRA 适配器

LoRA 微调的输出不是完整的 8B 参数模型，而是一组**增量权重**（约几十 MB）。

加载时，将适配器叠加到原始基础模型之上即可，无需重新下载完整模型权重。

> **第4讲预告**：保存的适配器将在下一讲中被重新加载，用于构建面向整个视频语料库的自动化术语处理流水线。

In [ ]:
# ===== 第9章：保存 LoRA 适配器与分词器 =====
import os

ADAPTER_SAVE_PATH = './qlora_adapter_lecture3'

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f'适配器已保存至：{ADAPTER_SAVE_PATH}')
print('保存内容：')
for fname in sorted(os.listdir(ADAPTER_SAVE_PATH)):
    fsize = os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, fname))
    print(f'  {fname:<40}  {fsize / 1e6:.2f} MB')

# 第9章（续）  下载适配器到本地

Colab 会话结束后，所有运行时文件都会丢失。运行下方单元格，将保存好的适配器打包成 zip 文件并下载到本地，以便在第4讲或其他环境中重新加载。

> 下载完成后，将 zip 文件解压至项目的 `shared/weights/` 目录，与课程仓库保持一致。

In [ ]:
import shutil
from google.colab import files

# ADAPTER_SAVE_PATH 已在上方定义为 './qlora_adapter_lecture3'
archive_name = 'qlora_adapter_lecture3_archive'

# 将适配器文件夹打包为 zip 压缩文件
shutil.make_archive(archive_name, 'zip', ADAPTER_SAVE_PATH)

# 触发浏览器下载
files.download(f'{archive_name}.zip')

print(f"文件夹 '{ADAPTER_SAVE_PATH}' 已打包并以 '{archive_name}.zip' 下载。")

# 快速回顾（Quick Check）

**本讲完成了以下工作：**

- **QLoRA 加载**：用 4 位 NF4 量化将 LLaMA-3 8B 的显存需求压缩至约 5–6 GB，在 Colab 免费 GPU（T4）上可用
- **LoRA 注入**：冻结基础模型，仅训练注意力层的低秩矩阵（可训练参数 < 1%），在保留通用能力的同时实现领域适配
- **双检查点策略**：先验证基础模型可推理（检查点 1），再验证训练流程使损失下降（检查点 2），确保不在错误流程上浪费算力
- **可扩展模板**：第8章的代码框架可直接替换为真实的视频转写数据集；第4讲将在此适配器基础上构建完整的语料处理流水线

> 和前两讲的连贯性：L01 把转写文本变短变可读；L02 把画面与古籍对齐；**L03 让模型真正认识领域术语。**